# SpeechT5 + WavLM Fine-Tuning

This notebook fine-tunes **SpeechT5** (`microsoft/speecht5_vc`) to accept **WavLM hidden states**
as encoder input instead of raw audio, enabling a WavLM-based feature-space speech translation pipeline.

## Architecture
```
EN raw audio  ──► [WavLM Encoder (frozen)]  ──► EN hidden states (Seq, 768)
                                                        │
                                 [SpeechT5 Transformer (fine-tuned)] ──► DE mel spectrogram
                                                        │
                                             [HiFi-GAN Vocoder]  ──► DE audio
```

## Required Pre-Step
The preprocessed WavLM dataset must exist at:
```
datasets/processed_wavlm_en_de_v1/
    en/   ← WavLM hidden states (Seq, 768)
    de/   ← WavLM hidden states (Seq, 768)
```
Run `preprocess_wavlm.py` if the dataset is not yet generated.

> **Note on the target representation:**  
> The dataset stores WavLM features for both sides. The `SpeechT5WavLMDataset`
> class in `model.py` handles the fallback to mel-spectrogram extraction if the
> target is raw audio. The fine-tuning loss is computed against SpeechT5's
> mel-spectrogram decoder output — this is what teaches the model to bridge
> the WavLM representation into something the trained decoder can process.


## 1. Setup & Imports

In [1]:
import os
import sys

# Add project root to Python path so that dataset_loader and encoders are importable.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

from model import SpeechT5WavLM
import dataset_loader
from datasets import load_from_disk
import numpy as np
from IPython.display import Audio, display

print(f"Project root: {project_root}")
print(f"Datasets directory: {dataset_loader.DATASETS_DIR}")

/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model
Datasets directory: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets


## 2. Configuration

Edit the constants below to control training behaviour.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Path to the preprocessed WavLM dataset produced by preprocess_wavlm.py.
# ─────────────────────────────────────────────────────────────────────────────
PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_wavlm_en_de_v8")

# ─────────────────────────────────────────────────────────────────────────────
# Training hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS        = 5       # Total training epochs
BATCH_SIZE    = 1        # Keep small — WavLM sequences are variable length
LEARNING_RATE = 2e-5     # Low LR for stable fine-tuning of pre-trained weights

# ─────────────────────────────────────────────────────────────────────────────
# Output checkpoint directory
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "speecht5_wavlm_en_de_v2026_05_11"

print(f"Preprocessed data: {PREPROCESSED_PATH}")
print(f"Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"Checkpoint will be saved to: {CHECKPOINT_DIR}")

Preprocessed data: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v8
Epochs: 5  |  Batch size: 1  |  LR: 2e-05
Checkpoint will be saved to: speecht5_wavlm_en_de_v2026_05_11


## 4. Initialise the Model

`SpeechT5WavLM` loads:
- **SpeechT5** (`microsoft/speecht5_vc`) — the transformer to fine-tune
- **HiFi-GAN** (`microsoft/speecht5_hifigan`) — the vocoder (frozen, used at inference)

WavLM itself is **not** loaded here; it was already used during preprocessing to produce the
hidden states stored in the dataset. It is only needed again at inference time
(handled by `run_inference`).

In [7]:
model = SpeechT5WavLM()
model.load("speecht5_wavlm_en_de_v2026_05_11_001")
print(f"\nModel loaded on device: {model.device}")

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda
Loading WavLM projection layer...

Model loaded on device: cuda


## 5. Extract Target Speaker Embedding

The SpeechT5 decoder needs an **x-vector speaker embedding** to condition its output voice.

`get_speaker_embedding` streams one sample from Google FLEURS for the target language and
computes the x-vector using the SpeechBrain classifier. The embedding is saved alongside
the model checkpoint.

In [8]:
model.get_speaker_embedding(target_lang="de")

Initializing X-Vector classifier for embedding extraction...


Extracting target speaker embedding...
Embedding extracted successfully.


## 6. Fine-Tune

Training will:
1. Load the preprocessed WavLM dataset.
2. Freeze SpeechT5's CNN feature encoder (only transformer layers are trained).
3. Feed WavLM hidden states directly into SpeechT5's transformer encoder.
4. Compute the spectrogram loss against the decoder's mel output.
5. Save a checkpoint every 5 epochs. On `KeyboardInterrupt` progress is saved
   automatically to `speecht5_wavlm_interrupted`.

In [5]:
model.fine_tune(
    preprocessed_path=PREPROCESSED_PATH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

Starting WavLM+SpeechT5 fine-tuning (Hybrid Architecture).
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v8...
Loaded 1433 aligned (source, target) pairs.
Dataset columns: ['id', 'input_values', 'labels', 'speaker_embeddings']
Freezing SpeechT5 CNN feature encoder (not used in hybrid path).


Epoch 1/5:   0%|          | 4/1433 [00:02<16:55,  1.41it/s, loss=4.3398, l1_pre=2.1907, l1_post=2.1472]
Traceback (most recent call last):
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/SpeechT5WavLM/model.py", line 527, in fine_tune
    (loss / GRAD_ACCUM_STEPS).backward()
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/_tensor.py", line 625, in backward
    torch.autograd.backward(
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/autograd/__init__.py", line 354, in backward
    _engine_run_backward(
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/autograd/graph.py", line 841, in _engine_run_ba


Error during training: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.69 GiB of which 2.19 MiB is free. Including non-PyTorch memory, this process has 3.68 GiB memory in use. Of the allocated memory 3.53 GiB is allocated by PyTorch, and 41.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Saved to 'speecht5_wavlm_error_mid_train'. Exiting safely.


## 7. Save the Final Checkpoint

In [ ]:
model.save(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")

## 8. Quick Inference Test

Load a raw EN clip from the raw dataset and run it through the full pipeline
(WavLM → fine-tuned SpeechT5 transformer → HiFi-GAN) to hear if the model is learning.

In [9]:
source_lang = "en"
target_lang = "de"
# model.load("speecht5_wavlm_en_de_v2026_05_11_001")
# Load a raw EN sample from the seamless_align dataset for inference.
print("Loading a raw EN sample for inference test...")
raw = dataset_loader.load_data(
    lang=[source_lang],
    split="train",
    dataset=["fleurs"],
    num_samples=1000,
)
sample = raw[source_lang][3]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f"Input audio: {len(audio_array)/sr:.2f}s @ {sr}Hz")
print("Original EN audio:")
display(Audio(data=audio_array, rate=sr))

# Run the full pipeline: raw audio → WavLM → fine-tuned SpeechT5 → HiFi-GAN → audio
print("\nRunning fine-tuned inference...")
result = model.run_inference(
    audio_array=audio_array,
    sampling_rate=sr,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=2.0,
)

out_audio = result['audio']['array']
out_sr    = result['audio']['sampling_rate']

print(f"Output audio: {len(out_audio)/out_sr:.2f}s @ {out_sr}Hz")
print("\nGenerated DE audio (after fine-tuning):")
display(Audio(data=out_audio, rate=out_sr))

Loading a raw EN sample for inference test...
Loading google/fleurs (en_us) from local storage...
Validating en (audio & uniqueness)...
Final Count: 1476 common valid samples.
Input audio: 5.40s @ 16000Hz
Original EN audio:



Running fine-tuned inference...
Loading WavLM (microsoft/wavlm-base-plus) extractor for inference...
Output audio: 8.61s @ 16000Hz

Generated DE audio (after fine-tuning):


## Bonus: Resume Training from a Checkpoint

If training was interrupted, load the saved checkpoint and continue.

In [ ]:
# ── Uncomment to resume from a saved checkpoint ──────────────────────────────
# RESUME_CHECKPOINT = "speecht5_wavlm_en_de_v2"
# model = SpeechT5WavLM()
# model.load(RESUME_CHECKPOINT)
# model.fine_tune(
#     preprocessed_path=PREPROCESSED_PATH,
#     epochs=EPOCHS,
#     learning_rate=LEARNING_RATE,
#     batch_size=BATCH_SIZE,
# )

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda
Starting WavLM+SpeechT5 fine-tuning (Hybrid Architecture).
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2...
Loaded 13438 aligned (source, target) pairs.
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels']
Freezing SpeechT5 CNN feature encoder (not used in hybrid path).


Epoch 1/20: 100%|██████████| 13438/13438 [19:06<00:00, 11.72it/s, loss=6.2368, l1_pre=3.3518, l1_post=2.8850] 


Epoch 1/20  Avg Loss: 6.7098


Epoch 2/20: 100%|██████████| 13438/13438 [19:08<00:00, 11.70it/s, loss=7.2392, l1_pre=3.8928, l1_post=3.3465] 


Epoch 2/20  Avg Loss: 6.6340


Epoch 3/20:   5%|▌         | 675/13438 [00:58<18:18, 11.62it/s, loss=6.8683, l1_pre=3.6925, l1_post=3.1758] 
/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 1876}
  warnings.warn(



Training interrupted! Saving current progress...
Saved to 'speecht5_wavlm_interrupted'. Exiting safely.


## Fine-tune the Vocoder (HiFi-GAN)

If you want to improve the audio quality further, you can fine-tune the vocoder using adversarial training. This freezes the SpeechT5 and WavLM backbones and exclusively updates the HiFi-GAN parameters.

In [ ]:

VOCODER_CHECKPOINT_DIR = "speecht5_wavlm_vocoder_en_de_v1"

# ─────────────────────────────────────────────────────────────────────────────
# Vocoder Fine-tuning
# ─────────────────────────────────────────────────────────────────────────────

VOCODER_PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_vocoder_en_de_v2")
VOCODER_EPOCHS = 10
VOCODER_LEARNING_RATE = 2e-5
VOCODER_BATCH_SIZE = 1

model.fine_tune_vocoder(
    preprocessed_path=VOCODER_PREPROCESSED_PATH,
    epochs=VOCODER_EPOCHS,
    learning_rate=VOCODER_LEARNING_RATE,
    batch_size=VOCODER_BATCH_SIZE
)



In [ ]:
model.save(VOCODER_CHECKPOINT_DIR)

## 9. Validate Vocoder Dataset

Listen to the ground truth and predicted spectrograms from the vocoder dataset to ensure preprocessing and fine-tuning are working as expected.

In [5]:
import torch
from IPython.display import Audio, display
from datasets import load_from_disk
VOCODER_PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_vocoder_en_de_v2")
# 1. Load the vocoder dataset
vocoder_ds = load_from_disk(VOCODER_PREPROCESSED_PATH)
print(f"Loaded vocoder dataset from: {VOCODER_PREPROCESSED_PATH}")
print(f"Dataset columns: {vocoder_ds.column_names}")

# 2. Select samples to validate
num_samples = 3
model.vocoder.eval()

for i in range(min(num_samples, len(vocoder_ds))):
    sample = vocoder_ds[i]
    print(f"\n{'='*60}\nSample {i}\n{'='*60}")
    
    # A. Ground Truth German Audio (Target)
    if "target_waveform" in sample:
        print("Target Audio (Ground Truth):")
        display(Audio(sample["target_waveform"], rate=16000))
        
    # B. Reconstructed from Ground Truth Mel (Labels)
    with torch.no_grad():
        labels = torch.tensor(sample["labels"], dtype=torch.float32).unsqueeze(0).to(model.device)
        if labels.shape[-1] != 80 and labels.shape[-2] == 80:
            labels = labels.transpose(1, 2)
        
        wav_gt = model.vocoder(labels).cpu().squeeze().numpy()
        print("Vocoder Reconstruction (from Ground Truth Mel):")
        display(Audio(wav_gt, rate=16000))
        
    # C. Reconstructed from Predicted Mel (SpeechT5 output)
    if "predicted_mel" in sample:
        with torch.no_grad():
            pred_mel = torch.tensor(sample["predicted_mel"], dtype=torch.float32).unsqueeze(0).to(model.device)
            if pred_mel.shape[-1] != 80 and pred_mel.shape[-2] == 80:
                pred_mel = pred_mel.transpose(1, 2)
            
            wav_pred = model.vocoder(pred_mel).cpu().squeeze().numpy()
            print("Vocoder Reconstruction (from Predicted Mel):")
            display(Audio(wav_pred, rate=16000))

Loaded vocoder dataset from: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_vocoder_en_de_v2
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels', 'target_waveform', 'predicted_mel']

Sample 0
Target Audio (Ground Truth):


Vocoder Reconstruction (from Ground Truth Mel):


Vocoder Reconstruction (from Predicted Mel):



Sample 1
Target Audio (Ground Truth):


Vocoder Reconstruction (from Ground Truth Mel):


Vocoder Reconstruction (from Predicted Mel):



Sample 2
Target Audio (Ground Truth):


Vocoder Reconstruction (from Ground Truth Mel):


Vocoder Reconstruction (from Predicted Mel):


## 10. Inspect Seamless Align Dataset

Show the columns and a sample row from the `seamless_align` dataset to understand its structure.

In [6]:
# Load a small sample of the seamless align dataset
seamless_ds_dict = dataset_loader.load_data(
    lang=["en", "de"], 
    dataset=["seamless_align"], 
    num_samples=100,
    split="train"
)

for lang, ds in seamless_ds_dict.items():
    if ds is not None:
        print(f"\nLanguage: {lang}")
        print(f"Columns: {ds.column_names}")
        # Print first row summary
        sample = ds[0]
        print(f"Sample data summary:")
        for col in ds.column_names:
            val = sample[col]
            if col == 'audio' or col == 'input_values' or col == 'labels' or col == 'target_waveform' or col == 'predicted_mel':
                if isinstance(val, dict) and 'array' in val:
                    print(f"  {col}: Audio dict with array of length {len(val['array'])}")
                elif hasattr(val, 'shape'):
                    print(f"  {col}: Tensor/Array with shape {val.shape}")
                else:
                    print(f"  {col}: <large data>")
            else:
                print(f"  {col}: {val}")

Loading jhu-clsp/seamless-align-expressive (deA-enA) from local storage: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/seamless_align/deA-enA/train...


Map: 100%|██████████| 100/100 [00:03<00:00, 27.11 examples/s]


Loading jhu-clsp/seamless-align-expressive (deA-enA) from local storage: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/seamless_align/deA-enA/train...


Map: 100%|██████████| 100/100 [00:01<00:00, 92.95 examples/s]


Validating en (checking audio & uniqueness)...


Validating en (num_proc=4): 100%|██████████| 100/100 [00:04<00:00, 23.56 examples/s]


  100 valid unique IDs found.
Validating de (checking audio & uniqueness)...


Validating de (num_proc=4): 100%|██████████| 100/100 [00:03<00:00, 25.32 examples/s]


  98 valid unique IDs found.
Final Count: 98 common valid samples.

Language: en
Columns: ['id', 'audio', 'language', 'gender', 'transcription']
Sample data summary:
  id: 5
  audio: Audio dict with array of length 67968
  language: en
  gender: unknown
  transcription: 

Language: de
Columns: ['id', 'audio', 'language', 'gender', 'transcription']
Sample data summary:
  id: 5
  audio: Audio dict with array of length 66240
  language: de
  gender: unknown
  transcription: 


## 11. Run Preprocessing (Both Directions)

These cells invoke `preprocess_vocoder_data.py` for both the **EN→DE** and **DE→EN** directions.
LID filtering is automatic — no extra flags needed. Run them sequentially.

Each run produces a clean, language-verified dataset:
- `datasets/processed_vocoder_en_de_v3/`
- `datasets/processed_vocoder_de_en_v3/`

### A. English → German

In [ ]:
import subprocess, sys

script = "preprocess_vocoder_data.py"

result = subprocess.run(
    [sys.executable, script, "--source", "en", "--target", "de", "--num-samples", "15000"],
    capture_output=False   # stream stdout/stderr live
)
print(f"Exit code: {result.returncode}")

### B. German → English

In [ ]:
import subprocess, sys

script = "preprocess_vocoder_data.py"

result = subprocess.run(
    [sys.executable, script, "--source", "de", "--target", "en", "--num-samples", "15000"],
    capture_output=False   # stream stdout/stderr live
)
print(f"Exit code: {result.returncode}")

## 12. Train Both Directional Models

Once preprocessing is complete for both directions, load each preprocessed dataset
and call `model.fine_tune()` and then `model.fine_tune_vocoder()` for each direction separately.

In [ ]:
# ──────────────────────────────────────────────
# English → German Model
# ──────────────────────────────────────────────
EN_DE_PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_vocoder_en_de_v3")
EN_DE_CHECKPOINT_DIR    = "speecht5_wavlm_en_de_v3"

model_en_de = SpeechT5WavLM()
model_en_de.fine_tune(
    preprocessed_path=EN_DE_PREPROCESSED_PATH,
    epochs=30,
    learning_rate=1e-4,
    batch_size=8,
)
model_en_de.save(EN_DE_CHECKPOINT_DIR)

In [ ]:
# ──────────────────────────────────────────────
# German → English Model
# ──────────────────────────────────────────────
DE_EN_PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_vocoder_de_en_v3")
DE_EN_CHECKPOINT_DIR    = "speecht5_wavlm_de_en_v1"

model_de_en = SpeechT5WavLM()
model_de_en.fine_tune(
    preprocessed_path=DE_EN_PREPROCESSED_PATH,
    epochs=30,
    learning_rate=1e-4,
    batch_size=8,
)
model_de_en.save(DE_EN_CHECKPOINT_DIR)

## 9. Diagnostic Check: Teacher-Forced Forward Pass

This cell runs a standard training forward pass (using teacher forcing) and passes the predicted mel-spectrogram to the vocoder. If it sounds like speech, the cross-attention alignment is working, and the previous mode-collapse was solely due to the inference loop lacking dropout. If it sounds like a continuous drone, cross-attention alignment has failed.

In [6]:
import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers.models.speecht5.modeling_speecht5 import shift_spectrograms_right
from IPython.display import Audio, display
from model import SpeechT5WavLMDataset, wavlm_speecht5_collate_fn

# 1. Put model in eval mode
model.model.eval()
model.vocoder.eval()
model.wavlm_proj.eval()

# 2. Load dataset and get one batch
paired_ds = load_from_disk(PREPROCESSED_PATH)
train_dataset = SpeechT5WavLMDataset(paired_ds, model.processor, model.target_embeddings)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=wavlm_speecht5_collate_fn)

batch = next(iter(train_loader))
input_values, attention_mask, labels, speaker_embeddings = batch
input_values = input_values.to(model.device)
attention_mask = attention_mask.to(model.device).long()
labels = labels.to(model.device)
speaker_embeddings = speaker_embeddings.to(model.device)

print("Running Teacher-Forced Forward Pass...")
with torch.no_grad():
    # Encode WavLM states
    encoder_out = model._encode_wavlm_states(input_values, attention_mask)
    
    # Shift targets for teacher forcing
    decoder_input_values, decoder_attention_mask = shift_spectrograms_right(
        labels, model.model.config.reduction_factor, None
    )
    
    # Forward pass through SpeechT5
    outputs = model.model.speecht5(
        encoder_outputs=encoder_out,
        attention_mask=attention_mask,
        decoder_input_values=decoder_input_values,
        decoder_attention_mask=decoder_attention_mask,
        speaker_embeddings=speaker_embeddings,
        use_cache=False,
        return_dict=True,
    )
    
    # Postnet projection
    _, outputs_after_postnet, _ = model.model.speech_decoder_postnet(outputs.last_hidden_state)
    
    # Take the predicted mel from the first item in the batch
    predicted_mel = outputs_after_postnet[0]
    
    # Generate audio with vocoder
    print("Vocoding predicted mel-spectrogram...")
    speech = model.vocoder(predicted_mel).squeeze()
    
audio_arr = speech.cpu().numpy()
print("Done! Listen below:")
display(Audio(audio_arr, rate=16000))


Running Teacher-Forced Forward Pass...
Vocoding predicted mel-spectrogram...
Done! Listen below:
